In [2]:
from brian2 import *
import os
import sys
sys.path = [p for p in sys.path if 'Neuron and Synapse Models' not in p and 'Tools' not in p]
os.chdir(os.path.dirname(os.getcwd()))  # Change to the parent directory
sys.path.append('Neuron and Synapse Models')
sys.path.append('Tools')

from neuronModels import *
from ringAttractorClass import *
from plottingTools import *
from utils import compute_firing_rate

import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, interactive_output, FloatSlider, IntSlider, Dropdown, HTML, FloatText, Label, ToggleButton

# Simulation parameters
defaultclock.dt = 0.1*ms

In [3]:
# Code block to allow easy passing of intital values
values_flag = True
if values_flag:
    init_values = {
        'num_neurons': 120,
        'tau_val': 10,
        'sigma_noise_val': 0.1,
        'stimulus_center': 3.14,
        'stimulus_width': 0.5,
        'I0_val': 30,
        'sigma_exc_val': 0.125,
        'sigma_inh_val': 0.25,
        'g_exc_val': 0.875,
        'g_inh_val': -0.475,
        'g_cosine_val': 0.1,
        'w_inh_val':-0.555,
        'duration_val': 2.0,
        'velocity_duration_val': 0.5,
        'connectivity_profile': 'cosine'
    }
    
# Good Connectivity parameters - Mexican hat
# sigma_exc = 0.0875
# sigma_inh = 0.25
# g_exc = 1.0*mV
# g_inh = -0.475*mV

In [4]:
# Create sliders for parameters
num_neurons_slider = IntSlider(
    min=50,
    max=200, 
    step=10, 
    value=init_values['num_neurons'] if values_flag else 120, 
    description='Number of Neurons:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

tau_slider = FloatSlider(
    min=1, 
    max=20, 
    step=1, 
    value=init_values['tau_val'] if values_flag else 10, 
    description='Tau (ms):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_noise_slider = FloatSlider(
    min=0.1, 
    max=5, 
    step=0.1, 
    value=init_values['sigma_noise_val'] if values_flag else 1, 
    description='Noise Sigma (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_center_slider = FloatSlider(
    min=0, 
    max=2*pi, 
    step=0.1, 
    value=init_values['stimulus_center'] if values_flag else 0, 
    description='Stimulus Center (rad):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_width_slider = FloatSlider(
    min=0.1, 
    max=2.0, 
    step=0.1, 
    value=init_values['stimulus_width'] if values_flag else 0.5, 
    description='Stimulus Width:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

I0_slider = FloatSlider(
    min=0, 
    max=50, 
    step=5, 
    value=init_values['I0_val'] if values_flag else 30, 
    description='Input Amplitude (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_exc_slider = FloatSlider(
    min=0.05,
    max=2.0,
    step=0.01,
    value=init_values['sigma_exc_val'] if values_flag else 0.0875,
    description='Sigma Excitatory:',
    continuous_update=False,
    style={'description_width': '150px'},
)

sigma_inh_slider = FloatSlider(
    min=0.1,
    max=2.0,
    step=0.01,
    value=init_values['sigma_inh_val'] if values_flag else 0.25,
    description='Sigma Inhibitory:',
    continuous_update=False,
    style={'description_width': '150px'},
)

g_exc_slider = FloatSlider(
    min=0.5,
    max=3.0,
    step=0.01,
    value=init_values['g_exc_val'] if values_flag else 1.0,
    description='Gain Excitatory (mV):',
    continuous_update=False,
    style={'description_width': '150px'},
)

g_inh_slider = FloatSlider(
    min=-3.0,
    max=-0.1,
    step=0.01,
    value=init_values['g_inh_val'] if values_flag else -0.475,
    description='Gain Inhibitory (mV):',
    continuous_update=False,
    style={'description_width': '150px'},
)

g_cosine_slider = FloatSlider(
    min=0.0,
    max=10.0,
    step=0.005,
    value=init_values['g_cosine_val'] if values_flag else 1.0,  # conditional value for cosine gain
    description='Gain Cosine (mV):',
    continuous_update=False,
    style={'description_width': '150px'}
)

velocity_input_slider = FloatSlider(
    min=-2.0, 
    max=2.0, 
    step=0.1, 
    value=0.0, 
    description='Velocity Input (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

w_inh_slider = FloatSlider(
    min=-3.00, 
    max=-0.01, 
    step=0.01, 
    value=init_values['w_inh_val'] if values_flag else -0.52705411, 
    description='Global Inhibition (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

duration_box = FloatText(
    value=init_values['duration_val'] if values_flag else 2,
    description='Simulation Duration (s):',
    style={'description_width': '150px'}
)

input_duration_box = FloatText(
    value=0.5,
    description='Input Duration (s):',
    style={'description_width': '150px'}
)

velocity_duration_box = FloatText(
    value=init_values['velocity_duration_val'] if values_flag else 0.5,
    description='Velocity Duration (s):',
    style={'description_width': '150px'}
)

profile_dropdown = Dropdown(
    options=['mexican_hat', 'gaussian', 'cosine'],
    value=init_values['connectivity_profile'] if values_flag else 'cosine',
    description='Synapse Profile:',
    style={'description_width': '150px'},
)

angles_dropdown = Dropdown(
    options=[ 'degrees', 'radians', 'radians (symbolic)'],
    value='degrees',
    description='Angular Representation:',
    style={'description_width': '150px'},
)

autapse_button = ToggleButton(
    value = False,
    description='Autapse',
    tooltip='Allows autapse connections',
    button_style=''
)

global_inh_button = ToggleButton(
    value = True,
    description='Global Inhibitory Neuron',
    tooltip='Adds a global inhibitory neuron',
    button_style='',
    layout = Layout(width='auto', height='auto')
)

# Create dictionary of widgets
widgets = {
    'num_neurons': num_neurons_slider,
    'tau_val': tau_slider,
    'sigma_noise_val': sigma_noise_slider,
    'stimulus_center': stimulus_center_slider,
    'stimulus_width': stimulus_width_slider,
    'I0_val': I0_slider,
    'sigma_exc_val': sigma_exc_slider,
    'sigma_inh_val': sigma_inh_slider,
    'g_exc_val': g_exc_slider,
    'g_inh_val': g_inh_slider,
    'g_cosine_val': g_cosine_slider,
    'duration_val': duration_box,
    'input_duration_val': input_duration_box,
    'velocity_duration_val': velocity_duration_box,
    'syn_profile': profile_dropdown,
    'ticks_angles': angles_dropdown,
    'autapse': autapse_button,
    'global_inh': global_inh_button,
    'velocity_input': velocity_input_slider,
    'w_inh_val': w_inh_slider
}

In [5]:
# Updated interactive_simulator to accept velocity_duration_val
def interactive_simulator(num_neurons, tau_val, sigma_noise_val,
                          stimulus_center, stimulus_width, I0_val, 
                          sigma_exc_val, sigma_inh_val, g_exc_val, g_inh_val,
                          g_cosine_val, velocity_input, w_inh_val,
                          duration_val, input_duration_val, velocity_duration_val,
                          autapse, global_inh, syn_profile, ticks_angles):
    
    glob_inh_flag = global_inh   # use value from widget
    
    # Clear any previous figures
    plt.close('all')
    
    # Convert slider values to Brian units
    tau = tau_val * ms
    sigma_noise = sigma_noise_val * mV
    V_rest = -70 * mV
    I0 = I0_val * mV
    sim_duration = duration_val*second
    g_exc = g_exc_val * mV
    g_inh = g_inh_val * mV
    g_cosine = g_cosine_val * mV
    w_inh = w_inh_val * mV
    velocity_duration = velocity_duration_val  # Use the parameter from the widget
    
    # Define neuron positions
    positions = linspace(0, 2*pi, num_neurons, endpoint=False)
    
    # Calculate external input
    d = np.angle(np.exp(1j * (positions - stimulus_center)))
    I_ext_array = I0 * np.exp(-(d**2) / (2 * stimulus_width**2))
    
    # Set up neuron model
    neuron_eq = Equations(LIF_xi_vel_eq, tau=tau, V_rest=V_rest, sigma_noise=sigma_noise)
    
    # Set up ring attractor
    Vth = -48 * mV
    V_reset = -80 * mV
    refractory_period = 5 * ms
    
    # Create the ring attractor network
    ringAttractor = RingAttractor(neuron_eq, 
                         num_neurons, 
                         Vth, V_reset, refractory_period,
                         syn_profile=syn_profile,
                         autapse=autapse,
                         glob_inh=glob_inh_flag, w_inh=w_inh,
                         g_cosine=g_cosine,
                         sigma_exc=sigma_exc_val, sigma_inh=sigma_inh_val,
                         g_exc=g_exc, g_inh=g_inh)
    
    # Set external input
    ringAttractor.ring_pool.I_ext = I_ext_array
    ringAttractor.ring_pool.I_vel = 0.0*volt
    
        
    # Setup monitors
    spikemon = SpikeMonitor(ringAttractor.ring_pool)
    statemon = StateMonitor(ringAttractor.ring_pool, 'V', record=True)
    inputmon = StateMonitor(ringAttractor.ring_pool, 'I_ext', record=True)
    
    #+---------------------------------------------------------------------------+
    #|                             Network Operations                            |
    #+---------------------------------------------------------------------------+
    # Clipping - Reverse Potential Behaviour: Define a network operation to enforce the lower bound on the membrane potential
    @network_operation(dt=defaultclock.dt)
    def enforce_lower_bound():
        # Using the built-in clip function (from numpy)
        ringAttractor.ring_pool.V[:] = clip(ringAttractor.ring_pool.V[:], V_reset, inf*volt)
    
   
    # Set of Brian objects to be added to the network
    localObjects = [enforce_lower_bound,
                    spikemon, statemon, inputmon]
    
    if glob_inh_flag:
        statemon_inh = StateMonitor(ringAttractor.glob_inh_neuron, 'V', record=True)
        spikemon_inh = SpikeMonitor(ringAttractor.glob_inh_neuron)
        localObjects.extend([statemon_inh, spikemon_inh])
        
    
    net = Network(ringAttractor.BrianObjects+localObjects)
    
    
        
    input_on = input_duration_val * second
    input_off = input_on
    velocity_on = velocity_duration * second
    end_duration = sim_duration - input_on - input_off - velocity_on

    # Run simulation
    net.run(input_on)
    
    # Turn off input for the second half
    ringAttractor.ring_pool.I_ext = I_ext_array * 0
    net.run(input_off)
    
    # Turn on velocity input
    ringAttractor.ring_synapses_asym.vel_in = velocity_input
      
    net.run(velocity_on)
    
    # Turn off velocity input
    ringAttractor.ring_synapses_asym.vel_in = 0.0
    
    # Turn off velocity input and run for the rest of the duration
    net.run(end_duration)
    

    
    #+---------------------------------------------------------------------------+
    #|                           Plotting the Results                            |
    #+---------------------------------------------------------------------------+

    # Create a figure with 6 subplots arranged in 3 rows and 2 columns
    fig = plt.figure(figsize=(15, 15))

    # 1. Input Current Plot
    ax1 = fig.add_subplot(3, 2, 1)
    ax1.plot(positions/(2*pi), I_ext_array/mV)
    ax1.set_title('Input Current')
    ax1.set_xlabel('Position (rad)')
    ax1.set_ylabel('Current (mV)')

    # 2. Raster Plot
    ax2 = fig.add_subplot(3, 2, 2)
    raster_plot(spikemon, ax=ax2, stim_periods=(0*second, input_on),
                stim_display_method='highlight', duration=sim_duration, num_neurons=num_neurons, y_axisFull=True)

    # 3. Firing Rate Profile Plot
    ax3 = fig.add_subplot(3, 2, 3)
    firing_rate, _ = firing_rate_profile(spikemon, positions/(2*pi), input_off, ax=ax3)

    # 4. Polar Plot of the Population Vector Average (PVA)
    ax4 = fig.add_subplot(3, 2, 4, projection='polar')
    polar_plot_PVA(firing_rate, positions, scale=1.2, ax=ax4)

    # 5. Time-Resolved PVA Plot
    ax5 = fig.add_subplot(3, 2, 5)
    _, _ = time_resolved_PVA(spikemon, positions, sim_duration, num_neurons, window_size=50*ms,
                             ax=ax5, color_windows=True, stim_periods=(0*second, input_on))

    # 6. Membrane potential traces
    ax6 = fig.add_subplot(3, 2, 6)
    membrane_potential_traces(statemon, sim_duration, Vth = Vth, ax=ax6)
    
    # if glob_inh_flag:
    #     plt.figure()
    #     plt.plot(statemon_inh.t/ms, statemon_inh.V[0]/mV)
    #     plt.title('Inhibitory Neuron Membrane Potential')
    #     plt.xlabel('Time (ms)')
    #     plt.ylabel('Membrane Potential (mV)')
        
    #     plt.figure()
    #     plt.plot(spikemon_inh.t/ms, spikemon_inh.i, '.k')
    #     plt.title('Inhibitory Neuron Spikes')
    #     plt.xlabel('Time (ms)')
    #     plt.ylabel('Neuron Index')
        
    plt.tight_layout()
    plt.show()

In [6]:
# # Define common style and layout settings for sliders
# common_style = {'description_width': '150px'}
# common_layout = Layout(width='300px', margin='10px auto')

# Create the parameter boxes with descriptive titles

box_neurons = VBox([
    HTML(value="<b>Neurons Parameters:</b>"),
    num_neurons_slider,
    tau_slider,
    sigma_noise_slider
], layout=Layout(width='25%', align_items='center'))

box_input = VBox([
    HTML(value="<b>Input Parameters:</b>"),
    stimulus_center_slider,
    stimulus_width_slider,
    I0_slider,
    velocity_input_slider
], layout=Layout(width='25%', align_items='center'))

checkbox_subBox = HBox([
    autapse_button,
    global_inh_button
], layout=Layout(align_items='center'))

box_connectivity = VBox([
    HTML(value="<b>Connectivity Profile Parameters:</b>"),
    checkbox_subBox,
    profile_dropdown
], layout=Layout(width='25%', align_items='center'))

def update_connectivity_box(change):
    if change['new'] == 'cosine':
        box_connectivity.children = [
            HTML(value="<b>Connectivity Profile Parameters:</b>"),
            checkbox_subBox,
            profile_dropdown,
            g_cosine_slider
        ]
    elif change['new'] == 'gaussian':   
        box_connectivity.children = [
            HTML(value="<b>Connectivity Profile Parameters:</b>"),
            checkbox_subBox,
            profile_dropdown
        ]
    else:
        box_connectivity.children = [
            HTML(value="<b>Connectivity Profile Parameters:</b>"),
            checkbox_subBox,
            profile_dropdown,
            sigma_exc_slider,
            sigma_inh_slider,
            g_exc_slider,
            g_inh_slider
        ]
    if global_inh_button.value:
            box_connectivity.children = list(box_connectivity.children) + [w_inh_slider]

# Add observer for global inhibitory neuron toggle
def update_for_global_inh(change):
    # Call update_connectivity_box to refresh UI with current synapse profile
    update_connectivity_box({'new': profile_dropdown.value})

global_inh_button.observe(update_for_global_inh, names='value')
profile_dropdown.observe(update_connectivity_box, names='value')
update_connectivity_box({'new': profile_dropdown.value})

box_simulation = VBox([
    HTML(value="<b>Simulation Parameters:</b>"),
    input_duration_box,
    velocity_duration_box,
    duration_box,
    angles_dropdown
], layout=Layout(width='25%', align_items='center'))

controls = HBox(
    [box_neurons, box_input, box_connectivity, box_simulation],
    layout=Layout(justify_content='center', margin='20px')
)

out = interactive_output(interactive_simulator, widgets)
dashboard = VBox(
    [controls, out],
    layout=Layout(align_items='stretch', justify_content='space-around')
)

display(dashboard)